# Projet ML — Maintenance prédictive véhicules

Ce notebook est organisé **fonction par fonction** pour Google Colab.

Objectif :
- charger le dataset `vehicle_maintenance_data.csv`
- nettoyer les données
- créer de meilleures variables
- éviter l’overfitting
- comparer plusieurs modèles
- choisir le meilleur modèle robuste
- optimiser le seuil de décision
- générer une recommandation automatique
- sauvegarder le modèle pour le backend

## 1. Installation des librairies nécessaires

Cette cellule installe `imbalanced-learn`, utile pour utiliser `SMOTE` dans un pipeline.

In [ ]:
!pip install -q imbalanced-learn

## 2. Imports

On importe les librairies nécessaires pour le traitement des données, la visualisation, le machine learning et la sauvegarde du modèle.

In [ ]:

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    learning_curve
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    auc,
    average_precision_score,
    brier_score_loss
)

from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

import joblib

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")


## 3. Chargement du dataset

Cette fonction cherche le fichier localement.  
Si le fichier n’existe pas, elle demande de l’importer dans Google Colab.

In [ ]:
def load_vehicle_data(filename="vehicle_maintenance_data.csv"):
    possible_paths = [
        filename,
        "./" + filename,
        "/content/" + filename,
        "/mnt/data/" + filename
    ]

    for path in possible_paths:
        if Path(path).exists():
            print("Fichier trouvé :", path)
            return pd.read_csv(path)

    from google.colab import files
    print("Fichier non trouvé. Importez le fichier :", filename)
    uploaded = files.upload()

    first_file = list(uploaded.keys())[0]
    print("Fichier importé :", first_file)

    return pd.read_csv(first_file)

## 4. Nettoyage des données

Cette fonction :
- supprime les doublons ;
- vérifie que la cible existe ;
- remplace les valeurs manquantes numériques par la médiane ;
- remplace les valeurs manquantes catégorielles par `Unknown`.

In [ ]:
def clean_vehicle_dataset(df):
    df = df.copy()

    df = df.drop_duplicates()

    if "Need_Maintenance" not in df.columns:
        raise ValueError("La colonne cible Need_Maintenance est absente.")

    df["Need_Maintenance"] = df["Need_Maintenance"].astype(int)

    numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

    for col in numeric_cols:
        if col != "Need_Maintenance":
            df[col] = df[col].fillna(df[col].median())

    categorical_cols = df.select_dtypes(include=["object", "category"]).columns

    for col in categorical_cols:
        df[col] = df[col].fillna("Unknown")

    print("Nettoyage terminé.")
    print("Shape après nettoyage :", df.shape)

    return df

## 5. Analyse rapide du dataset

Cette fonction affiche :
- les premières lignes ;
- les statistiques ;
- les valeurs manquantes ;
- la distribution de la cible.

In [ ]:

def add_bar_labels(ax, values, suffix=""):
    """Ajoute la valeur au-dessus de chaque barre."""
    maximum = max(values) if len(values) else 1

    for patch, value in zip(ax.patches, values):
        ax.annotate(
            f"{value}{suffix}",
            (patch.get_x() + patch.get_width() / 2, patch.get_height()),
            ha="center",
            va="bottom",
            xytext=(0, 5),
            textcoords="offset points",
            fontsize=10,
            fontweight="bold"
        )

    ax.set_ylim(0, maximum * 1.18)


def quick_eda(df, target="Need_Maintenance"):
    print("Dimensions du dataset :", df.shape)
    display(df.head())

    print("\nStatistiques descriptives :")
    display(df.describe(include="all").T)

    print("\nValeurs manquantes par colonne :")
    missing = df.isnull().sum().sort_values(ascending=False)
    display(missing.to_frame("valeurs_manquantes"))

    if target not in df.columns:
        return

    counts = df[target].value_counts().sort_index()
    ratios = df[target].value_counts(normalize=True).sort_index() * 100

    distribution = pd.DataFrame({
        "classe": counts.index,
        "signification": [
            "Pas de maintenance" if value == 0 else "Maintenance nécessaire"
            for value in counts.index
        ],
        "nombre": counts.values,
        "pourcentage": ratios.values
    })

    print("\nDistribution de la variable cible :")
    display(distribution)

    labels = [
        "Pas de maintenance" if value == 0 else "Maintenance nécessaire"
        for value in counts.index
    ]

    plt.figure(figsize=(9, 5))
    ax = plt.gca()
    ax.bar(labels, counts.values)
    add_bar_labels(ax, counts.values)
    ax.set_title("Répartition initiale des classes")
    ax.set_xlabel("Classe réelle")
    ax.set_ylabel("Nombre de véhicules")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(9, 5))
    ax = plt.gca()
    ax.bar(labels, ratios.values)
    add_bar_labels(ax, [round(v, 2) for v in ratios.values], suffix="%")
    ax.set_title("Répartition initiale des classes en pourcentage")
    ax.set_xlabel("Classe réelle")
    ax.set_ylabel("Pourcentage")
    ax.set_ylim(0, max(ratios.values) * 1.22)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

    imbalance_ratio = counts.max() / max(counts.min(), 1)
    print(f"Ratio de déséquilibre : {imbalance_ratio:.2f}")
    if imbalance_ratio >= 1.5:
        print(
            "Interprétation : les classes sont déséquilibrées. "
            "Une méthode comme SMOTE ou class_weight='balanced' peut être utile."
        )
    else:
        print("Interprétation : le déséquilibre des classes reste limité.")


## 6. Feature engineering amélioré

Cette fonction crée des variables plus utiles pour la maintenance prédictive :
- nombre de jours depuis le dernier service ;
- jours restants avant expiration de la garantie ;
- kilométrage par année ;
- problèmes par service ;
- écart de kilométrage depuis le dernier service ;
- intensité d’utilisation ;
- véhicule ancien avec kilométrage élevé ;
- garantie expirée.

In [ ]:
def advanced_feature_engineering(data, reference_date=None):
    data = data.copy()

    if "Last_Service_Date" in data.columns:
        data["Last_Service_Date"] = pd.to_datetime(
            data["Last_Service_Date"],
            errors="coerce"
        )

    if "Warranty_Expiry_Date" in data.columns:
        data["Warranty_Expiry_Date"] = pd.to_datetime(
            data["Warranty_Expiry_Date"],
            errors="coerce"
        )

    if reference_date is None:
        if "Last_Service_Date" in data.columns and data["Last_Service_Date"].notna().any():
            reference_date = data["Last_Service_Date"].max() + pd.Timedelta(days=1)
        else:
            reference_date = pd.Timestamp.today()

    reference_date = pd.Timestamp(reference_date)

    if "Last_Service_Date" in data.columns:
        data["Days_Since_Last_Service"] = (
            reference_date - data["Last_Service_Date"]
        ).dt.days

        data["Days_Since_Last_Service"] = data["Days_Since_Last_Service"].fillna(
            data["Days_Since_Last_Service"].median()
        )

    if "Warranty_Expiry_Date" in data.columns:
        data["Warranty_Remaining_Days"] = (
            data["Warranty_Expiry_Date"] - reference_date
        ).dt.days

        data["Warranty_Remaining_Days"] = data["Warranty_Remaining_Days"].fillna(
            data["Warranty_Remaining_Days"].median()
        )

        data["Warranty_Expired"] = (
            data["Warranty_Remaining_Days"] < 0
        ).astype(int)

    if "Mileage" in data.columns and "Vehicle_Age" in data.columns:
        data["Mileage_Per_Year"] = data["Mileage"] / data["Vehicle_Age"].replace(0, 1)

        data["Old_And_High_Mileage"] = (
            (data["Vehicle_Age"] >= 8) &
            (data["Mileage"] >= data["Mileage"].median())
        ).astype(int)

    if "Reported_Issues" in data.columns and "Service_History" in data.columns:
        data["Issues_Per_Service"] = data["Reported_Issues"] / (
            data["Service_History"] + 1
        )

    if "Odometer_Reading" in data.columns and "Last_Service_Mileage" in data.columns:
        data["Mileage_Service_Gap"] = (
            data["Odometer_Reading"] - data["Last_Service_Mileage"]
        )

    if "Mileage" in data.columns and "Days_Since_Last_Service" in data.columns:
        data["Vehicle_Usage_Intensity"] = data["Mileage"] / (
            data["Days_Since_Last_Service"].replace(0, 1)
        )

    data = data.drop(
        columns=["Last_Service_Date", "Warranty_Expiry_Date"],
        errors="ignore"
    )

    data = data.replace([np.inf, -np.inf], np.nan)

    numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns
    for col in numeric_cols:
        data[col] = data[col].fillna(data[col].median())

    categorical_cols = data.select_dtypes(include=["object", "category"]).columns
    for col in categorical_cols:
        data[col] = data[col].fillna("Unknown")

    return data

## 7. Détection des colonnes trop directes

Certaines colonnes peuvent donner une réponse trop facile au modèle, par exemple :
- `Brake_Condition`
- `Battery_Status`
- `Tire_Condition`
- `Reported_Issues`
- `Maintenance_History`

On les supprime dans le scénario robuste pour éviter un modèle trop parfait mais peu réaliste.

In [ ]:
def detect_direct_or_leaky_columns(df, target="Need_Maintenance", threshold=0.85):
    suspicious_cols = []

    manual_direct_cols = [
        "Maintenance_History",
        "Reported_Issues",
        "Tire_Condition",
        "Brake_Condition",
        "Battery_Status"
    ]

    for col in manual_direct_cols:
        if col in df.columns:
            suspicious_cols.append(col)

    numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

    for col in numeric_cols:
        if col != target:
            corr = abs(df[col].corr(df[target]))
            if pd.notna(corr) and corr >= threshold:
                suspicious_cols.append(col)

    suspicious_cols = list(set(suspicious_cols))

    print("Colonnes suspectes / trop directes détectées :")
    print(suspicious_cols)

    return suspicious_cols

## 8. Préprocessing

Cette fonction prépare les colonnes :
- les colonnes catégorielles sont transformées avec `OneHotEncoder` ;
- les colonnes numériques peuvent être standardisées avec `StandardScaler`.

In [ ]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_better_preprocessor(X, scale_numeric=False):
    categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    numeric_cols = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

    numeric_transformer = StandardScaler() if scale_numeric else "passthrough"

    preprocessor = ColumnTransformer(
        transformers=[
            ("categorical", make_ohe(), categorical_cols),
            ("numeric", numeric_transformer, numeric_cols),
        ],
        remainder="drop"
    )

    return preprocessor, categorical_cols, numeric_cols

## 9. Comparaison des modèles

Cette fonction compare plusieurs modèles avec validation croisée stratifiée :

- DummyClassifier : baseline seulement ;
- Logistic Regression + SMOTE ;
- Random Forest régularisé ;
- Extra Trees régularisé ;
- Gradient Boosting ;
- HistGradientBoosting.

On privilégie `balanced_accuracy`, `roc_auc` et `f1`.

In [ ]:

def compare_better_models(X_train, y_train):
    preprocessor_tree, _, _ = build_better_preprocessor(
        X_train,
        scale_numeric=False
    )
    preprocessor_scaled, _, _ = build_better_preprocessor(
        X_train,
        scale_numeric=True
    )

    models = {
        "Dummy_Baseline": Pipeline(steps=[
            ("preprocess", preprocessor_tree),
            ("model", DummyClassifier(strategy="most_frequent"))
        ]),

        "LogisticRegression_SMOTE": ImbPipeline(steps=[
            ("preprocess", preprocessor_scaled),
            ("smote", SMOTE(random_state=42)),
            ("model", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            ))
        ]),

        "RandomForest_Tuned": Pipeline(steps=[
            ("preprocess", preprocessor_tree),
            ("model", RandomForestClassifier(
                n_estimators=300,
                max_depth=10,
                min_samples_split=30,
                min_samples_leaf=15,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            ))
        ]),

        "ExtraTrees_Tuned": Pipeline(steps=[
            ("preprocess", preprocessor_tree),
            ("model", ExtraTreesClassifier(
                n_estimators=300,
                max_depth=10,
                min_samples_split=30,
                min_samples_leaf=15,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            ))
        ]),

        "GradientBoosting": Pipeline(steps=[
            ("preprocess", preprocessor_tree),
            ("model", GradientBoostingClassifier(
                n_estimators=150,
                learning_rate=0.05,
                max_depth=3,
                random_state=42
            ))
        ]),

        "HistGradientBoosting": Pipeline(steps=[
            ("preprocess", preprocessor_tree),
            ("model", HistGradientBoostingClassifier(
                max_iter=200,
                learning_rate=0.05,
                max_leaf_nodes=20,
                random_state=42
            ))
        ])
    }

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    scoring = {
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    }

    rows = []
    cv_details = {}

    for name, model in models.items():
        print("Évaluation :", name)

        scores = cross_validate(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
            return_train_score=False
        )

        cv_details[name] = {
            metric: scores[f"test_{metric}"]
            for metric in scoring
        }

        row = {"model": name}

        for metric in scoring:
            values = scores[f"test_{metric}"]
            row[metric] = values.mean()
            row[f"{metric}_std"] = values.std()

        rows.append(row)

    results = pd.DataFrame(rows).sort_values(
        by=["balanced_accuracy", "roc_auc", "f1"],
        ascending=False
    ).reset_index(drop=True)

    return results, models, cv_details


## 10. Sélection du meilleur modèle

Le modèle Dummy est utilisé uniquement pour comparaison.  
Il ne doit jamais être sélectionné comme modèle final.

In [ ]:
def select_best_model(results, models):
    candidates = results[results["model"] != "Dummy_Baseline"].copy()

    candidates = candidates.sort_values(
        by=["balanced_accuracy", "roc_auc", "f1"],
        ascending=False
    )

    best_name = candidates.iloc[0]["model"]
    best_model = models[best_name]

    print("Meilleur modèle sélectionné :", best_name)
    display(candidates.head())

    return best_name, best_model

## 11. Optimisation du seuil de décision

Le seuil 0.5 n’est pas toujours optimal.  
Cette fonction cherche le seuil qui maximise la métrique choisie, par défaut `balanced_accuracy`.

In [ ]:

def find_best_threshold(y_true, y_proba, metric="balanced_accuracy"):
    thresholds = np.arange(0.05, 0.96, 0.01)
    rows = []

    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)

        rows.append({
            "threshold": threshold,
            "accuracy": accuracy_score(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "precision": precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            "recall": recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            "f1": f1_score(
                y_true,
                y_pred,
                zero_division=0
            )
        })

    threshold_df = pd.DataFrame(rows)

    best_row = threshold_df.sort_values(
        by=[metric, "f1", "recall"],
        ascending=False
    ).iloc[0]

    best_threshold = float(best_row["threshold"])

    print("Meilleur seuil :", round(best_threshold, 3))
    print("Métrique optimisée :", metric)
    display(best_row.to_frame().T)

    plt.figure(figsize=(11, 6))
    ax = plt.gca()

    for column, label in [
        ("precision", "Précision"),
        ("recall", "Rappel"),
        ("f1", "F1-score"),
        ("balanced_accuracy", "Balanced accuracy")
    ]:
        ax.plot(
            threshold_df["threshold"],
            threshold_df[column],
            linewidth=2,
            label=label
        )

    ax.axvline(
        best_threshold,
        linestyle="--",
        linewidth=2,
        label=f"Seuil optimal = {best_threshold:.2f}"
    )
    ax.scatter(
        [best_threshold],
        [best_row[metric]],
        s=90,
        zorder=5
    )

    ax.set_xlabel("Seuil de décision")
    ax.set_ylabel("Score")
    ax.set_title("Influence du seuil sur les performances du modèle")
    ax.set_xlim(0.05, 0.95)
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.25)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

    return best_threshold, threshold_df


## 12. Évaluation finale

Cette fonction évalue le modèle final avec le seuil optimisé.

In [ ]:

def final_evaluation(model, X_test, y_test, threshold):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "average_precision": average_precision_score(y_test, y_proba),
        "brier_score": brier_score_loss(y_test, y_proba)
    }

    print("Évaluation finale avec seuil optimisé")
    display(pd.DataFrame([metrics]).T.rename(columns={0: "score"}))

    report = classification_report(
        y_test,
        y_pred,
        target_names=[
            "Pas de maintenance",
            "Maintenance nécessaire"
        ],
        output_dict=True,
        zero_division=0
    )

    print("\nRapport de classification détaillé :")
    display(pd.DataFrame(report).T)

    return metrics, y_pred, y_proba


## 13. Graphes d’évaluation

Courbe ROC et courbe Precision-Recall.

In [ ]:

def plot_confusion_matrix_detailed(y_true, y_pred, normalized=False):
    matrix = confusion_matrix(
        y_true,
        y_pred,
        normalize="true" if normalized else None
    )

    labels = [
        "Pas de maintenance",
        "Maintenance nécessaire"
    ]

    plt.figure(figsize=(8, 6))
    ax = plt.gca()
    image = ax.imshow(matrix)

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Classe prédite")
    ax.set_ylabel("Classe réelle")

    title = (
        "Matrice de confusion normalisée"
        if normalized
        else "Matrice de confusion — nombres"
    )
    ax.set_title(title)

    threshold_text = matrix.max() / 2 if matrix.size else 0

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = matrix[i, j]
            text = f"{value:.1%}" if normalized else f"{int(value)}"
            ax.text(
                j,
                i,
                text,
                ha="center",
                va="center",
                fontsize=14,
                fontweight="bold",
                color="white" if value > threshold_text else "black"
            )

    plt.colorbar(image, ax=ax)
    plt.xticks(rotation=12)
    plt.tight_layout()
    plt.show()


def plot_roc_curve_detailed(y_true, y_proba, title="Modèle final"):
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    roc_auc = auc(fpr, tpr)

    youden_index = tpr - fpr
    best_index = int(np.argmax(youden_index))

    plt.figure(figsize=(9, 6))
    ax = plt.gca()
    ax.plot(
        fpr,
        tpr,
        linewidth=2.5,
        label=f"{title} — AUC = {roc_auc:.4f}"
    )
    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Modèle aléatoire — AUC = 0.50"
    )
    ax.scatter(
        fpr[best_index],
        tpr[best_index],
        s=90,
        zorder=5,
        label=(
            "Meilleur compromis ROC "
            f"(seuil ≈ {thresholds[best_index]:.2f})"
        )
    )

    ax.set_xlabel("Taux de faux positifs")
    ax.set_ylabel("Taux de vrais positifs")
    ax.set_title("Courbe ROC — capacité de discrimination")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


def plot_precision_recall_detailed(y_true, y_proba, title="Modèle final"):
    precision, recall, thresholds = precision_recall_curve(
        y_true,
        y_proba
    )
    pr_auc = auc(recall, precision)
    average_precision = average_precision_score(y_true, y_proba)
    baseline = float(np.mean(y_true))

    plt.figure(figsize=(9, 6))
    ax = plt.gca()
    ax.plot(
        recall,
        precision,
        linewidth=2.5,
        label=(
            f"{title} — PR AUC = {pr_auc:.4f} — "
            f"AP = {average_precision:.4f}"
        )
    )
    ax.axhline(
        baseline,
        linestyle="--",
        label=f"Référence classe positive = {baseline:.2%}"
    )

    ax.set_xlabel("Rappel")
    ax.set_ylabel("Précision")
    ax.set_title("Courbe précision-rappel")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()


def plot_probability_distributions(y_true, y_proba, threshold):
    y_array = np.asarray(y_true)

    plt.figure(figsize=(10, 6))
    ax = plt.gca()

    ax.hist(
        y_proba[y_array == 0],
        bins=20,
        alpha=0.65,
        label="Pas de maintenance"
    )
    ax.hist(
        y_proba[y_array == 1],
        bins=20,
        alpha=0.65,
        label="Maintenance nécessaire"
    )
    ax.axvline(
        threshold,
        linestyle="--",
        linewidth=2,
        label=f"Seuil de décision = {threshold:.2f}"
    )

    ax.set_xlabel("Probabilité prédite de maintenance")
    ax.set_ylabel("Nombre de véhicules")
    ax.set_title("Distribution des probabilités prédites par classe réelle")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_calibration_detailed(y_true, y_proba):
    prob_true, prob_pred = calibration_curve(
        y_true,
        y_proba,
        n_bins=10,
        strategy="quantile"
    )

    score = brier_score_loss(y_true, y_proba)

    plt.figure(figsize=(8, 6))
    ax = plt.gca()
    ax.plot(
        prob_pred,
        prob_true,
        marker="o",
        linewidth=2,
        label=f"Modèle — score de Brier = {score:.4f}"
    )
    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Calibration parfaite"
    )

    ax.set_xlabel("Probabilité moyenne prédite")
    ax.set_ylabel("Fréquence réellement observée")
    ax.set_title("Courbe de calibration des probabilités")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_final_curves(y_test, y_proba, title="Modèle final"):
    plot_roc_curve_detailed(y_test, y_proba, title)
    plot_precision_recall_detailed(y_test, y_proba, title)



## 13 bis. Visualisations avancées et interprétation

Cette section ajoute des diagrammes plus complets pour expliquer :

- le déséquilibre initial des classes ;
- l’effet de SMOTE sur les données d’entraînement ;
- la stabilité des modèles en validation croisée ;
- la matrice de confusion brute et normalisée ;
- les courbes ROC et précision-rappel ;
- la distribution des probabilités ;
- la calibration ;
- la courbe d’apprentissage ;
- l’importance des variables.

> **Important :** SMOTE est appliqué uniquement aux données d’entraînement.  
> Le jeu de test ne doit jamais être équilibré artificiellement.


In [ ]:

def plot_smote_balance(X_train, y_train):
    """
    Illustre l'effet de SMOTE sur les données d'entraînement uniquement.
    Cette visualisation ne modifie pas X_train ni y_train.
    """
    preprocessor, _, _ = build_better_preprocessor(
        X_train,
        scale_numeric=True
    )

    X_prepared = preprocessor.fit_transform(X_train)
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(
        X_prepared,
        y_train
    )

    before_counts = pd.Series(y_train).value_counts().sort_index()
    after_counts = pd.Series(y_resampled).value_counts().sort_index()

    labels = [
        "Pas de maintenance",
        "Maintenance nécessaire"
    ]

    plt.figure(figsize=(9, 5))
    ax = plt.gca()
    ax.bar(labels, before_counts.reindex([0, 1], fill_value=0).values)
    add_bar_labels(
        ax,
        before_counts.reindex([0, 1], fill_value=0).values
    )
    ax.set_title("Classes avant SMOTE — données d'entraînement")
    ax.set_ylabel("Nombre d'observations")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(9, 5))
    ax = plt.gca()
    ax.bar(labels, after_counts.reindex([0, 1], fill_value=0).values)
    add_bar_labels(
        ax,
        after_counts.reindex([0, 1], fill_value=0).values
    )
    ax.set_title("Classes après SMOTE — données d'entraînement")
    ax.set_ylabel("Nombre d'observations")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

    comparison = pd.DataFrame({
        "classe": labels,
        "avant_SMOTE": before_counts.reindex([0, 1], fill_value=0).values,
        "apres_SMOTE": after_counts.reindex([0, 1], fill_value=0).values
    })

    display(comparison)

    print(
        "Interprétation : SMOTE crée des observations synthétiques "
        "pour la classe minoritaire jusqu'à obtenir des classes équilibrées."
    )


def plot_model_metric_comparison(results):
    metrics_to_plot = [
        "balanced_accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ]

    labels = {
        "balanced_accuracy": "Balanced accuracy",
        "precision": "Précision",
        "recall": "Rappel",
        "f1": "F1-score",
        "roc_auc": "ROC AUC"
    }

    ordered = results.sort_values(
        "balanced_accuracy",
        ascending=True
    )

    for metric in metrics_to_plot:
        plt.figure(figsize=(10, 6))
        ax = plt.gca()

        values = ordered[metric].values
        errors = ordered.get(
            f"{metric}_std",
            pd.Series(np.zeros(len(ordered)))
        ).values

        ax.barh(
            ordered["model"],
            values,
            xerr=errors,
            capsize=4
        )

        for index, value in enumerate(values):
            ax.text(
                min(value + 0.012, 0.98),
                index,
                f"{value:.3f}",
                va="center",
                fontweight="bold"
            )

        ax.set_xlim(0, 1.05)
        ax.set_xlabel(labels[metric])
        ax.set_ylabel("Modèle")
        ax.set_title(
            f"Comparaison des modèles — {labels[metric]} "
            "(moyenne ± écart-type)"
        )
        ax.grid(axis="x", alpha=0.25)
        plt.tight_layout()
        plt.show()


def plot_cv_boxplots(cv_details, metric="balanced_accuracy"):
    data = []
    labels = []

    for model_name, metric_values in cv_details.items():
        data.append(metric_values[metric])
        labels.append(model_name)

    plt.figure(figsize=(12, 6))
    ax = plt.gca()
    ax.boxplot(data, labels=labels, showmeans=True)
    ax.set_ylabel(metric.replace("_", " ").title())
    ax.set_title(
        f"Stabilité en validation croisée — {metric.replace('_', ' ').title()}"
    )
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()


def plot_learning_curve_detailed(
    estimator,
    X,
    y,
    scoring="balanced_accuracy"
):
    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    train_sizes, train_scores, validation_scores = learning_curve(
        clone(estimator),
        X,
        y,
        cv=cv,
        scoring=scoring,
        train_sizes=np.linspace(0.20, 1.00, 5),
        n_jobs=-1
    )

    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    validation_mean = validation_scores.mean(axis=1)
    validation_std = validation_scores.std(axis=1)

    plt.figure(figsize=(10, 6))
    ax = plt.gca()

    ax.plot(
        train_sizes,
        train_mean,
        marker="o",
        linewidth=2,
        label="Score entraînement"
    )
    ax.fill_between(
        train_sizes,
        train_mean - train_std,
        train_mean + train_std,
        alpha=0.15
    )

    ax.plot(
        train_sizes,
        validation_mean,
        marker="o",
        linewidth=2,
        label="Score validation"
    )
    ax.fill_between(
        train_sizes,
        validation_mean - validation_std,
        validation_mean + validation_std,
        alpha=0.15
    )

    ax.set_xlabel("Nombre d'observations d'entraînement")
    ax.set_ylabel(scoring.replace("_", " ").title())
    ax.set_title("Courbe d'apprentissage du modèle final")
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()

    gap = train_mean[-1] - validation_mean[-1]
    print(f"Écart final entraînement-validation : {gap:.4f}")

    if gap > 0.10:
        print(
            "Interprétation : écart important, un surapprentissage "
            "reste possible."
        )
    elif validation_mean[-1] < 0.65:
        print(
            "Interprétation : les deux scores restent faibles ; "
            "le modèle peut être en sous-apprentissage."
        )
    else:
        print(
            "Interprétation : les courbes indiquent une généralisation "
            "globalement correcte."
        )


def plot_permutation_importance_detailed(
    model,
    X_test,
    y_test,
    top_n=15
):
    importance = permutation_importance(
        model,
        X_test,
        y_test,
        scoring="balanced_accuracy",
        n_repeats=8,
        random_state=42,
        n_jobs=-1
    )

    importance_df = pd.DataFrame({
        "variable": X_test.columns,
        "importance": importance.importances_mean,
        "ecart_type": importance.importances_std
    }).sort_values(
        "importance",
        ascending=False
    ).head(top_n)

    plot_df = importance_df.sort_values(
        "importance",
        ascending=True
    )

    plt.figure(figsize=(10, 7))
    ax = plt.gca()
    ax.barh(
        plot_df["variable"],
        plot_df["importance"],
        xerr=plot_df["ecart_type"],
        capsize=3
    )
    ax.set_xlabel(
        "Diminution de la balanced accuracy après permutation"
    )
    ax.set_ylabel("Variable")
    ax.set_title(
        f"Importance des {top_n} variables principales"
    )
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()

    display(importance_df.reset_index(drop=True))


## 14. Recommandation métier

Cette fonction transforme la prédiction ML en recommandation compréhensible.

In [ ]:
def generate_maintenance_recommendation(row, probability):
    recommendations = []

    if probability >= 0.80:
        risk = "Risque critique"
        recommendations.append("Planifier une maintenance urgente.")
    elif probability >= 0.60:
        risk = "Risque élevé"
        recommendations.append("Planifier une inspection dans les prochains jours.")
    elif probability >= 0.40:
        risk = "Risque moyen"
        recommendations.append("Programmer un contrôle préventif.")
    else:
        risk = "Risque faible"
        recommendations.append("Continuer le suivi normal du véhicule.")

    if row.get("Mileage", 0) > 150000:
        recommendations.append("Contrôler les composants liés au kilométrage élevé.")

    if row.get("Vehicle_Age", 0) >= 8:
        recommendations.append("Vérifier les éléments sensibles liés à l'âge du véhicule.")

    if row.get("Reported_Issues", 0) >= 3:
        recommendations.append("Analyser les problèmes signalés par le conducteur.")

    if row.get("Tire_Condition") == "Worn Out":
        recommendations.append("Contrôler ou remplacer les pneus.")

    if row.get("Brake_Condition") == "Worn Out":
        recommendations.append("Contrôler ou remplacer le système de freinage.")

    if row.get("Battery_Status") == "Weak":
        recommendations.append("Tester ou remplacer la batterie.")

    return risk, " | ".join(recommendations)

## 15. Fonction finale de prédiction

Cette fonction retourne :
- probabilité de maintenance ;
- prédiction ;
- niveau de risque ;
- recommandation.

In [ ]:
def predict_vehicle_maintenance(model, raw_data, threshold, direct_cols):
    prepared_data = advanced_feature_engineering(raw_data)

    model_input = prepared_data.drop(columns=direct_cols, errors="ignore")

    probabilities = model.predict_proba(model_input)[:, 1]
    predictions = (probabilities >= threshold).astype(int)

    output = raw_data.copy().reset_index(drop=True)

    output["Maintenance_Probability"] = probabilities
    output["Need_Maintenance_Predicted"] = predictions

    risks = []
    recommendations = []

    for i in range(len(output)):
        risk, recommendation = generate_maintenance_recommendation(
            output.loc[i],
            probabilities[i]
        )

        risks.append(risk)
        recommendations.append(recommendation)

    output["Risk_Level"] = risks
    output["Recommendation"] = recommendations

    return output

## 16. Sauvegarde des fichiers pour le backend

On sauvegarde :
- le modèle ;
- le seuil optimisé ;
- les colonnes supprimées.

In [ ]:
def save_final_artifacts(model, threshold, direct_cols):
    joblib.dump(model, "vehicle_maintenance_model.pkl")
    joblib.dump(threshold, "vehicle_maintenance_threshold.pkl")
    joblib.dump(direct_cols, "vehicle_maintenance_removed_columns.pkl")

    print("Modèle sauvegardé : vehicle_maintenance_model.pkl")
    print("Seuil sauvegardé : vehicle_maintenance_threshold.pkl")
    print("Colonnes supprimées sauvegardées : vehicle_maintenance_removed_columns.pkl")

# Exécution complète du pipeline

Les cellules suivantes exécutent le projet complet.

## 17. Charger et préparer les données

In [ ]:
df = load_vehicle_data("vehicle_maintenance_data.csv")
df = clean_vehicle_dataset(df)

quick_eda(df)

df_model = advanced_feature_engineering(df)

X_full = df_model.drop("Need_Maintenance", axis=1)
y = df_model["Need_Maintenance"]

direct_cols_existing = detect_direct_or_leaky_columns(df_model)

X_robust = X_full.drop(columns=direct_cols_existing, errors="ignore")

print("X_full shape :", X_full.shape)
print("X_robust shape :", X_robust.shape)

## 18. Split train/test

On garde une séparation stratifiée pour conserver la même distribution de classes dans train et test.

In [ ]:
X_train_robust, X_test_robust, y_train, y_test = train_test_split(
    X_robust,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train :", X_train_robust.shape)
print("Test :", X_test_robust.shape)

print("\nContrôle de la répartition des classes dans le train :")
display(
    pd.DataFrame({
        "nombre": y_train.value_counts().sort_index(),
        "pourcentage": (
            y_train.value_counts(normalize=True).sort_index() * 100
        )
    })
)

plot_smote_balance(X_train_robust, y_train)


## 19. Comparer les modèles

In [ ]:

results, models, cv_details = compare_better_models(
    X_train_robust,
    y_train
)

print("Classement complet des modèles :")
display(results)

plot_model_metric_comparison(results)
plot_cv_boxplots(
    cv_details,
    metric="balanced_accuracy"
)
plot_cv_boxplots(
    cv_details,
    metric="roc_auc"
)


## 20. Sélectionner et entraîner le meilleur modèle

In [ ]:
best_name, final_model = select_best_model(results, models)

final_model.fit(X_train_robust, y_train)

print("Modèle final entraîné :", best_name)

## 21. Optimiser le seuil

In [ ]:
y_proba_default = final_model.predict_proba(X_test_robust)[:, 1]

best_threshold, threshold_df = find_best_threshold(
    y_test,
    y_proba_default,
    metric="balanced_accuracy"
)

## 22. Évaluation finale

In [ ]:

metrics, y_pred, y_proba = final_evaluation(
    final_model,
    X_test_robust,
    y_test,
    best_threshold
)

print("\n1. Matrice de confusion en nombres")
plot_confusion_matrix_detailed(
    y_test,
    y_pred,
    normalized=False
)

print("\n2. Matrice de confusion normalisée")
plot_confusion_matrix_detailed(
    y_test,
    y_pred,
    normalized=True
)

print("\n3. Courbe ROC")
plot_roc_curve_detailed(
    y_test,
    y_proba,
    title=best_name
)

print("\n4. Courbe précision-rappel")
plot_precision_recall_detailed(
    y_test,
    y_proba,
    title=best_name
)

print("\n5. Distribution des probabilités")
plot_probability_distributions(
    y_test,
    y_proba,
    best_threshold
)

print("\n6. Calibration des probabilités")
plot_calibration_detailed(
    y_test,
    y_proba
)

print("\n7. Courbe d'apprentissage")
plot_learning_curve_detailed(
    final_model,
    X_train_robust,
    y_train,
    scoring="balanced_accuracy"
)

print("\n8. Importance des variables")
plot_permutation_importance_detailed(
    final_model,
    X_test_robust,
    y_test,
    top_n=15
)



### Comment interpréter les diagrammes

**SMOTE**

- Le premier graphique présente la distribution réelle des classes dans le jeu d’entraînement.
- Le deuxième montre la distribution après création d’observations synthétiques.
- SMOTE ne doit être appliqué qu’au jeu d’entraînement.

**Validation croisée**

- La hauteur des barres représente la performance moyenne sur les cinq folds.
- Les barres d’erreur représentent l’écart-type.
- Un modèle performant et stable possède une moyenne élevée et un écart-type faible.
- Les boxplots permettent de vérifier qu’un bon score ne provient pas d’un seul découpage favorable.

**Matrice de confusion**

- Vrai négatif : véhicule correctement classé sans maintenance.
- Faux positif : maintenance annoncée alors qu’elle n’était pas nécessaire.
- Faux négatif : maintenance nécessaire mais non détectée.
- Vrai positif : besoin de maintenance correctement détecté.
- Dans ce projet, les faux négatifs sont particulièrement importants, car ils peuvent laisser circuler un véhicule à risque.

**Courbe ROC**

- Elle mesure la capacité générale du modèle à séparer les deux classes.
- Une AUC proche de 1 indique une bonne discrimination.
- La diagonale correspond à un modèle aléatoire.

**Courbe précision-rappel**

- Elle est particulièrement utile quand les classes sont déséquilibrées.
- Une précision élevée signifie que les alertes émises sont souvent correctes.
- Un rappel élevé signifie que peu de véhicules réellement à risque sont oubliés.

**Distribution des probabilités**

- Elle montre si le modèle sépare clairement les véhicules sains des véhicules nécessitant une maintenance.
- Un fort chevauchement entre les deux distributions indique des cas difficiles à distinguer.

**Calibration**

- Une courbe proche de la diagonale signifie que les probabilités sont fiables.
- Par exemple, parmi les véhicules annoncés à 70 %, environ 70 % devraient réellement nécessiter une maintenance.

**Courbe d’apprentissage**

- Un grand écart entre entraînement et validation peut indiquer un surapprentissage.
- Deux courbes faibles et proches peuvent indiquer un sous-apprentissage.
- Deux courbes élevées et proches indiquent généralement une bonne généralisation.

**Importance des variables**

- Une importance élevée signifie que la permutation de la variable dégrade fortement la performance.
- Cela montre quelles caractéristiques influencent le plus la prédiction finale.


## 23. Tester les recommandations

In [ ]:
sample = df.drop(columns=["Need_Maintenance"]).sample(10, random_state=42)

predictions = predict_vehicle_maintenance(
    final_model,
    sample,
    best_threshold,
    direct_cols_existing
)

cols_to_display = [
    "Vehicle_Model",
    "Mileage",
    "Vehicle_Age",
    "Maintenance_Probability",
    "Need_Maintenance_Predicted",
    "Risk_Level",
    "Recommendation"
]

available_cols = [c for c in cols_to_display if c in predictions.columns]

display(predictions[available_cols])


## 23 bis. Tableau de synthèse pour le rapport ou la soutenance

La cellule suivante produit automatiquement un résumé compact du modèle final.


In [ ]:

summary = pd.DataFrame({
    "Indicateur": [
        "Meilleur modèle",
        "Seuil optimal",
        "Accuracy",
        "Balanced accuracy",
        "Précision",
        "Rappel",
        "F1-score",
        "ROC AUC",
        "Average precision",
        "Score de Brier",
        "Nombre de variables utilisées",
        "Colonnes retirées"
    ],
    "Valeur": [
        best_name,
        round(best_threshold, 3),
        round(metrics["accuracy"], 4),
        round(metrics["balanced_accuracy"], 4),
        round(metrics["precision"], 4),
        round(metrics["recall"], 4),
        round(metrics["f1"], 4),
        round(metrics["roc_auc"], 4),
        round(metrics["average_precision"], 4),
        round(metrics["brier_score"], 4),
        X_train_robust.shape[1],
        ", ".join(direct_cols_existing)
    ]
})

display(summary)


## 24. Sauvegarder les fichiers

In [ ]:
save_final_artifacts(
    final_model,
    best_threshold,
    direct_cols_existing
)

## 25. Télécharger les fichiers depuis Colab

Exécute cette cellule si tu veux télécharger le modèle, le seuil et la liste des colonnes supprimées.

In [ ]:
from google.colab import files

files.download("vehicle_maintenance_model.pkl")
files.download("vehicle_maintenance_threshold.pkl")
files.download("vehicle_maintenance_removed_columns.pkl")